In [1]:
import re
import pandas as pd

from aumann_ratio_decomposition import (
    AumannShapleyRatioDecomposer,
    SDConfig,
    SubgroupLooper,
)

In [2]:
# ============================================
# 1. Toy data: Product × age × impact
# ============================================
data = [
    {"Product": "A", "age": 10, "impact": -50},
    {"Product": "A", "age": 20, "impact": -50},
    {"Product": "B", "age": 10, "impact": 50},
    {"Product": "B", "age": 20, "impact": -20},
    {"Product": "B", "age": 30, "impact": -45},
]

df_toy = pd.DataFrame(data)

In [5]:
# ============================================
# 2. Run Subgroup Discovery on `impact`
#    - target_col: impact
#    - features: Product, age
# ============================================

cfg_toy = SDConfig(
    target_col="impact",           # subgroupの評価対象
    use_abs_std_qf_numeric=True,   # 絶対値版 StandardQFNumeric
    depth=2,                       # 条件の最大個数（Product & age の組合せを見るので 2 で十分）
    alpha_start=0.1,
    alpha_decay=0.3,
    min_support=1,                 # サンプルが5件しかないので 1 まで許容
    ratio_threshold=0.0,           # ごく小さいデータなので制約は緩める
    max_loops=None,
    intervals_only=False,          # age を区間化しても、離散のままでもどちらでもOK
    nbins=3,                       # age のビン数（適当に少なめに）
)

runner_toy = SubgroupLooper(df_toy, cfg_toy)
df_toy_summary = runner_toy.run()

print(df_toy_summary)
df_toy_summary


[Loop 0] remaining=5, alpha=0.100
  -> picked 1 rows (23.26% of total target, 76.74% remain), condition: Product=='B' AND age==10
[Loop 1] remaining=4, alpha=0.370
  -> picked 1 rows (9.30% of total target, 67.44% remain), condition: Product=='B' AND age==20
[Loop 2] remaining=3, alpha=0.559
  -> picked 1 rows (20.93% of total target, 46.51% remain), condition: age==30
[Loop 3] remaining=2, alpha=0.691
  -> picked 2 rows (46.51% of total target, 0.00% remain), condition: Dataset
   rank                      rule  support  sum
0     1  Product=='B' AND age==10        1   50
1     2  Product=='B' AND age==20        1  -20
2     3                   age==30        1  -45
3     4                   Dataset        2 -100


,rank,rule,support,sum
0,1,Product=='B' AND age==10,1,50
1,2,Product=='B' AND age==20,1,-20
2,3,age==30,1,-45
3,4,Dataset,2,-100
